# Atividade prática III: Raízes de equações
**Cálculo Numérico — Prof.ª Raquel J. Lobosco**  
**Dupla: Daniel Cordazzo e Rodrigo Valle**

Implementação, comparação e visualização dos métodos da **bisseção**, de **Newton** e da **secante**, conforme `atividade_3.pdf`.

Ambiente: Python 3, compatível com Jupyter e Google Colab. Os arquivos de saída são gravados na pasta `resultados`. As funções e os valores iniciais são definidos em `problemas`; a tolerância e o limite de iterações, em `TOL` e `MAX_ITER`.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image, Markdown

TOL = 1e-6
MAX_ITER = 100
SAIDA = Path('resultados')
SAIDA.mkdir(exist_ok=True)
pd.set_option('display.precision', 10)

## 1. Implementações e critérios de parada

Na bisseção, calculamos $m=(a+b)/2$ e preservamos o subintervalo com troca de sinal. Para funções contínuas, isso mantém uma raiz no intervalo. Em Newton usamos $x_{k+1}=x_k-f(x_k)/f'(x_k)$. Na secante substituímos a derivada pela inclinação entre as duas últimas aproximações.

Adotamos **resíduo $|f(x)|<10^{-6}$** como critério de sucesso nos três métodos. Uma raiz inicial que já satisfaz esse critério retorna com zero iterações. Uma diferença pequena entre iterados, sozinha, não garante resíduo pequeno; por isso não é suficiente para declarar convergência. Na bisseção registramos também a meia largura do intervalo, que limita o erro em $x$ sob a hipótese de continuidade.

Usamos até 100 atualizações. Valores não finitos, avaliações inválidas, intervalos sem troca de sinal, derivada próxima de zero e denominador próximo de zero geram falhas explícitas. Para Newton, o limiar absoluto da derivada é $10^{-14}$. Na secante, o limiar do denominador é $10^{-14}\max(1,|f(x_0)|,|f(x_1)|)$. Esses limiares são apropriados à escala das funções desta atividade; problemas reescalados podem exigir ajuste.

Cada retorno é um dicionário com raiz, iterações, aproximações calculadas, convergência, motivo e histórico geométrico. As aproximações iniciais ficam no histórico e não contam como atualizações.

In [ ]:
def avaliar(f, x):
    # Recusa avaliações indefinidas ou não finitas.
    y = float(f(float(x)))
    if not math.isfinite(y):
        raise ValueError('Avaliação não finita')
    return y

def validar(tol, max_iter):
    if not math.isfinite(tol) or tol <= 0:
        raise ValueError('tol deve ser finita e positiva')
    if not isinstance(max_iter, int) or max_iter < 1:
        raise ValueError('max_iter deve ser inteiro positivo')

def retorno(x, hist, ok, motivo):
    return dict(raiz=x, iteracoes=len(hist),
                aproximacoes=[h['x'] for h in hist],
                convergiu=ok, motivo=motivo, historico=hist)

def bissecao(f, a, b, tol=1e-6, max_iter=100):
    validar(tol, max_iter)
    hist = []
    x = None
    try:
        if not (math.isfinite(a) and math.isfinite(b) and a < b):
            return retorno(x, hist, False, 'Intervalo inválido')
        fa, fb = avaliar(f, a), avaliar(f, b)
        for ponta, valor in [(a, fa), (b, fb)]:
            if abs(valor) < tol:
                return retorno(ponta, hist, True, 'Resíduo inicial < tol')
        if np.signbit(fa) == np.signbit(fb):
            return retorno(x, hist, False, 'Intervalo sem troca de sinal')
        for k in range(1, max_iter + 1):
            x = a + (b - a) / 2
            fx = avaliar(f, x)
            hist.append(dict(k=k, a=a, b=b, x=x, fx=fx, erro=(b-a)/2))
            if abs(fx) < tol:
                return retorno(x, hist, True, 'Resíduo < tol')
            if x == a or x == b:
                return retorno(x, hist, False, 'Estagnação em ponto flutuante')
            if np.signbit(fa) != np.signbit(fx):
                b, fb = x, fx
            else:
                a, fa = x, fx
    except (ValueError, OverflowError, ZeroDivisionError) as exc:
        return retorno(x, hist, False, str(exc))
    return retorno(x, hist, False, 'Limite de iterações')

def newton(f, df, x0, tol=1e-6, max_iter=100):
    validar(tol, max_iter)
    hist, x = [], float(x0)
    try:
        if not math.isfinite(x):
            return retorno(x, hist, False, 'Aproximação não finita')
        fx = avaliar(f, x)
        if abs(fx) < tol:
            return retorno(x, hist, True, 'Resíduo inicial < tol')
        for k in range(1, max_iter + 1):
            dfx = avaliar(df, x)
            if abs(dfx) <= 1e-14:
                return retorno(x, hist, False, 'Derivada próxima de zero')
            novo = x - fx / dfx
            if not math.isfinite(novo):
                return retorno(x, hist, False, 'Aproximação não finita')
            fn = avaliar(f, novo)
            hist.append(dict(k=k, anterior=x, f_anterior=fx, derivada=dfx,
                             x=novo, fx=fn, passo=abs(novo-x)))
            x, fx = novo, fn
            if abs(fx) < tol:
                return retorno(x, hist, True, 'Resíduo < tol')
    except (ValueError, OverflowError, ZeroDivisionError) as exc:
        return retorno(x, hist, False, str(exc))
    return retorno(x, hist, False, 'Limite de iterações')

def secante(f, x0, x1, tol=1e-6, max_iter=100):
    validar(tol, max_iter)
    hist = []
    x0, x1 = float(x0), float(x1)
    try:
        if not (math.isfinite(x0) and math.isfinite(x1)):
            return retorno(x1, hist, False, 'Aproximação não finita')
        f0, f1 = avaliar(f, x0), avaliar(f, x1)
        for x, fx in [(x0, f0), (x1, f1)]:
            if abs(fx) < tol:
                return retorno(x, hist, True, 'Resíduo inicial < tol')
        for k in range(1, max_iter + 1):
            denominador = f1 - f0
            if abs(denominador) <= 1e-14 * max(1.0, abs(f0), abs(f1)):
                return retorno(x1, hist, False, 'Denominador próximo de zero')
            novo = x1 - f1 * (x1-x0) / denominador
            if not math.isfinite(novo):
                return retorno(x1, hist, False, 'Aproximação não finita')
            fn = avaliar(f, novo)
            hist.append(dict(k=k, x0=x0, x1=x1, f0=f0, f1=f1,
                             x=novo, fx=fn, passo=abs(novo-x1)))
            x0, f0, x1, f1 = x1, f1, novo, fn
            if abs(fn) < tol:
                return retorno(x1, hist, True, 'Resíduo < tol')
    except (ValueError, OverflowError, ZeroDivisionError) as exc:
        return retorno(x1, hist, False, str(exc))
    return retorno(x1, hist, False, 'Limite de iterações')

## 2. Aplicação às quatro funções

Usamos exatamente os valores iniciais sugeridos. As derivadas são $f'_1(x)=2x$, $f'_2(x)=3x^2-1$, $f'_3(x)=-e^{-x}-1$ e $f'_4(x)=3x^2-2$.

Todos os intervalos propostos têm troca de sinal: os valores nos extremos são, respectivamente, $(-1,2)$, $(-2,4)$, $(1,e^{-1}-1)$ e $(-2,3)$. Como as funções são contínuas, a bisseção tem sua hipótese satisfeita. Para $f_1$, investigamos a raiz positiva; a raiz negativa também existe, mas está fora do intervalo escolhido.

In [ ]:
problemas = {
    'f1': dict(nome='x² − 2', f=lambda x: x*x-2, df=lambda x: 2*x,
               intervalo=(1., 2.), newton=1.5, secante=(1., 2.)),
    'f2': dict(nome='x³ − x − 2', f=lambda x: x**3-x-2, df=lambda x: 3*x*x-1,
               intervalo=(1., 2.), newton=1.5, secante=(1., 2.)),
    'f3': dict(nome='exp(−x) − x', f=lambda x: math.exp(-x)-x,
               df=lambda x: -math.exp(-x)-1,
               intervalo=(0., 1.), newton=0.5, secante=(0., 1.)),
    'f4': dict(nome='x³ − 2x + 2', f=lambda x: x**3-2*x+2,
               df=lambda x: 3*x*x-2,
               intervalo=(-2., -1.), newton=-2., secante=(-2., -1.)),
}
resultados, linhas = {}, []
for chave, p in problemas.items():
    casos = [
        ('Bisseção', p['intervalo'], bissecao(p['f'], *p['intervalo'], TOL, MAX_ITER)),
        ('Newton', p['newton'], newton(p['f'], p['df'], p['newton'], TOL, MAX_ITER)),
        ('Secante', p['secante'], secante(p['f'], *p['secante'], TOL, MAX_ITER)),
    ]
    for metodo, iniciais, r in casos:
        resultados[chave, metodo] = r
        linhas.append({
            'Função': f"{chave}: {p['nome']}", 'Método': metodo,
            'Valores iniciais': str(iniciais), 'Raiz aproximada': r['raiz'],
            '|f(x)| final': abs(p['f'](r['raiz'])) if r['raiz'] is not None else np.nan,
            'Iterações/resultado': f"{r['iteracoes']} / {'convergiu' if r['convergiu'] else 'falhou'}",
        })
tabela = pd.DataFrame(linhas)
display(tabela)
tabela.to_csv(SAIDA / 'tabela_resultados.csv', index=False, encoding='utf-8-sig')
tabela.to_html(SAIDA / 'tabela_resultados.html', index=False, float_format=lambda x: f'{x:.10g}')

In [ ]:
for chave, p in problemas.items():
    partes = [f"{m}: {resultados[chave, m]['iteracoes']} iterações"
              for m in ['Bisseção', 'Newton', 'Secante']]
    display(Markdown(f"**{chave}** — " + '; '.join(partes) + '.'))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (chave, p) in zip(axes.flat, problemas.items()):
    for metodo in ['Bisseção', 'Newton', 'Secante']:
        h = resultados[chave, metodo]['historico']
        ax.semilogy([t['k'] for t in h], [max(abs(t['fx']), 1e-16) for t in h],
                    '.-', label=metodo)
    ax.axhline(TOL, color='gray', linestyle='--', label='Tolerância')
    ax.set(title=f"{chave}: {p['nome']}", xlabel='Iteração', ylabel='Resíduo |f(x)|')
    ax.grid(alpha=.25)
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(SAIDA / 'comparacao_convergencia.png', dpi=150)
plt.show()

### Interpretação

As raízes encontradas são aproximadamente **1,414214**, **1,521380**, **0,567143** e **−1,769292**. Para os valores iniciais sugeridos, os três métodos atingem o resíduo solicitado.

A bisseção reduz a largura do intervalo pela metade a cada passo e apresenta convergência linear em termos desse limite de erro; o resíduo não precisa diminuir de forma monotônica. Newton tende a precisar de menos atualizações perto de raízes simples, com convergência local quadrática e uma derivada adequada. A secante dispensa a derivada e tem ordem local aproximadamente 1,618 sob as hipóteses usuais para raízes simples. Essas ordens são propriedades locais, não garantias para qualquer ponto inicial.

Contar iterações não equivale a medir tempo: Newton avalia a derivada a cada atualização, enquanto a secante reaproveita dois valores da função. A bisseção é previsível quando há continuidade e troca de sinal; Newton e secante podem sair do intervalo inicial e falhar.

## 3. Testes e influência da aproximação inicial

Em $f_4$, iniciar Newton em $x_0=0$ produz $0\to1\to0\to1\to\cdots$, pois $f_4(0)=2$, $f'_4(0)=-2$, $f_4(1)=1$ e $f'_4(1)=1$. O resíduo alterna entre 2 e 1, e o método falha após 100 atualizações. Esse ponto foi escolhido justamente para demonstrar que a convergência depende da aproximação inicial. Em $x_0=-2$, a mesma função converge.

In [ ]:
p = problemas['f4']
ciclo = newton(p['f'], p['df'], 0., TOL, MAX_ITER)
display(pd.DataFrame(ciclo['historico'][:8])[['k', 'anterior', 'x', 'fx']])
print('Newton em f4, x0=0:', ciclo['motivo'], '| iterações:', ciclo['iteracoes'])

falhas = {
    'Bisseção sem troca de sinal': bissecao(lambda x: x*x+1, -1, 1),
    'Newton com derivada nula': newton(lambda x: x*x-2, lambda x: 2*x, 0),
    'Newton com derivada quase nula': newton(lambda x: 1+x*1e-16, lambda x: 1e-16, 0),
    'Secante com denominador nulo': secante(lambda x: x*x-2, -1, 1),
    'Secante com denominador quase nulo': secante(lambda x: 1+x*1e-15, 0, 1),
    'Limite de iterações': bissecao(lambda x: x*x-2, 1, 2, max_iter=1),
    'Avaliação não finita': newton(lambda x: float('inf'), lambda x: 1, 0),
    'Newton em ciclo': ciclo,
}
display(pd.DataFrame([
    {'Teste': nome, 'Convergiu': r['convergiu'], 'Iterações': r['iteracoes'], 'Motivo': r['motivo']}
    for nome, r in falhas.items()
]))
assert all(not r['convergiu'] for r in falhas.values())
assert ciclo['iteracoes'] == MAX_ITER
assert ciclo['aproximacoes'][:4] == [1., 0., 1., 0.]
for (chave, metodo), r in resultados.items():
    assert r['convergiu'] and abs(problemas[chave]['f'](r['raiz'])) < TOL
    assert len(r['aproximacoes']) == r['iteracoes'] <= MAX_ITER
    if metodo == 'Bisseção':
        for h in r['historico']:
            assert problemas[chave]['f'](h['a']) * problemas[chave]['f'](h['b']) <= 0
            assert h['a'] <= h['x'] <= h['b']
assert bissecao(lambda x: x-1, 1, 2)['iteracoes'] == 0
assert newton(lambda x: x-1, lambda x: 1, 1)['iteracoes'] == 0
assert secante(lambda x: x-1, 1, 2)['iteracoes'] == 0
assert abs(resultados['f1', 'Newton']['raiz'] - math.sqrt(2)) < TOL
print('Todos os testes passaram.')

## 4. Animações geométricas

Geramos três combinações: **bisseção em $f_1$**, **Newton em $f_2$** e **secante em $f_3$**. Cada quadro corresponde a uma atualização real registrada no histórico. A marca vermelha representa a nova aproximação no eixo horizontal, e o ponto vermelho sobre a curva mostra seu resíduo. Pontos cinza mostram aproximações anteriores.

Na bisseção, a faixa azul é o intervalo usado no passo atual e a verde é o subintervalo preservado. O painel inferior amplia a redução do intervalo. Em Newton, a linha tracejada é a tangente no ponto anterior; na secante, passa pelos dois pontos anteriores. A interseção dessas retas com o eixo horizontal determina a nova aproximação.

In [ ]:
def animar(chave, metodo, arquivo):
    p = problemas[chave]
    hist = resultados[chave, metodo]['historico']
    a, b = p['intervalo']
    largura = b-a
    grade = np.linspace(a-.2*largura, b+.2*largura, 500)
    valores = [p['f'](x) for x in grade]
    fig, (ax, zoom) = plt.subplots(2, 1, figsize=(9, 6),
                                  gridspec_kw={'height_ratios': [4, 1]})

    def quadro(i):
        ax.clear()
        zoom.clear()
        h = hist[i]
        ax.plot(grade, valores, color='#2458a6', label='f(x)')
        ax.axhline(0, color='black', linewidth=.8)
        ax.scatter([t['x'] for t in hist[:i]], [t['fx'] for t in hist[:i]],
                   color='gray', s=20, label='Aproximações anteriores')
        if metodo == 'Bisseção':
            ax.axvspan(h['a'], h['b'], color='royalblue', alpha=.15, label='Intervalo atual')
            if p['f'](h['a']) * h['fx'] <= 0:
                na, nb = h['a'], h['x']
            else:
                na, nb = h['x'], h['b']
            ax.axvspan(na, nb, color='green', alpha=.25, label='Subintervalo preservado')
            for extremo in [h['a'], h['b']]:
                ax.axvline(extremo, color='royalblue', linestyle=':', linewidth=1)
            zoom.plot([h['a'], h['b']], [0, 0], 'o-', color='royalblue', linewidth=4)
            zoom.plot([na, nb], [0, 0], 'o-', color='green', linewidth=2)
            zoom.scatter([h['x']], [0], color='crimson', zorder=5)
            margem = (h['b']-h['a'])*.15
            zoom.set_xlim(h['a']-margem, h['b']+margem)
            zoom.set_title(f"Ampliação do intervalo: largura = {h['b']-h['a']:.3e}", fontsize=10)
            zoom.set_yticks([])
            zoom.ticklabel_format(axis='x', style='plain', useOffset=False)
        else:
            if metodo == 'Newton':
                linha = h['f_anterior'] + h['derivada']*(grade-h['anterior'])
                ax.scatter([h['anterior']], [h['f_anterior']], color='orange', s=65,
                           label='Ponto de tangência', zorder=5)
                legenda = 'Tangente'
            else:
                inclinacao = (h['f1']-h['f0'])/(h['x1']-h['x0'])
                linha = h['f1'] + inclinacao*(grade-h['x1'])
                ax.scatter([h['x0'], h['x1']], [h['f0'], h['f1']], color='orange',
                           s=65, label='Pontos da secante', zorder=5)
                legenda = 'Reta secante'
            ax.plot(grade, linha, '--', color='darkorange', label=legenda)
            zoom.axis('off')
            zoom.text(.02, .5, f"Passo |x novo − x anterior| = {h['passo']:.3e}",
                      transform=zoom.transAxes, fontsize=12)
        ax.scatter([h['x']], [0], marker='x', s=90, color='crimson', zorder=6,
                   label='Nova aproximação no eixo')
        ax.scatter([h['x']], [h['fx']], color='crimson', zorder=6)
        ax.plot([h['x'], h['x']], [0, h['fx']], ':', color='crimson')
        ax.set(xlim=(grade[0], grade[-1]), ylim=(min(valores)-.4, max(valores)+.4),
               xlabel='x', ylabel='f(x)',
               title=f"{metodo} — {chave}: {p['nome']}\nIteração {h['k']} / {len(hist)}"
                     f"   x = {h['x']:.10f}   |f(x)| = {abs(h['fx']):.3e}")
        ax.grid(alpha=.2)
        ax.legend(loc='best', fontsize=8)
        fig.tight_layout()

    animacao = FuncAnimation(fig, quadro, frames=len(hist), interval=850, repeat=True)
    destino = SAIDA / arquivo
    animacao.save(destino, writer=PillowWriter(fps=1.2), dpi=100)
    plt.close(fig)
    return destino

gifs = [
    animar('f1', 'Bisseção', 'bissecao_f1.gif'),
    animar('f2', 'Newton', 'newton_f2.gif'),
    animar('f3', 'Secante', 'secante_f3.gif'),
]
for gif in gifs:
    display(Markdown(f'### {gif.name}'))
    display(Image(filename=str(gif)))

## 5. Arquivos de resultados

A pasta `resultados` reúne os três GIFs, a tabela em CSV e HTML, o gráfico de convergência e o dashboard HTML. O arquivo `resultados_atividade_3.zip` reúne esses resultados.

In [ ]:
import html
import shutil

cartoes = ''.join(
    f'<section><h2>{html.escape(g.name)}</h2><img src="{g.name}" alt="Animação {g.stem}"></section>'
    for g in gifs
)
pagina = '''<!doctype html><html lang="pt-BR"><meta charset="utf-8">
<title>Atividade III — Cálculo Numérico</title>
<style>body{font-family:system-ui;max-width:1150px;margin:40px auto;padding:20px;background:#f4f7fb;color:#182c48}
section{background:white;padding:22px;margin:22px 0;border-radius:12px}img{max-width:100%}
table{border-collapse:collapse;width:100%;font-size:14px}td,th{padding:10px;border-bottom:1px solid #ddd;text-align:left}</style>
<h1>Raízes de equações — Atividade prática III</h1>
<p>Daniel Cordazzo e Rodrigo Valle · Prof.ª Raquel J. Lobosco</p>
<p>Tolerância: 10⁻⁶ · Máximo: 100 iterações · Parada: resíduo &lt; tolerância.</p>
<section><h2>Resultados</h2>'''
pagina += tabela.to_html(index=False, float_format=lambda x: f'{x:.10g}', border=0)
pagina += '</section><section><h2>Convergência</h2><img src="comparacao_convergencia.png" alt="Resíduo por iteração"></section>'
pagina += cartoes
pagina += '<section><h2>Influência do ponto inicial</h2><p>Newton em f4 converge com x0 = −2; com x0 = 0, entra no ciclo 0 → 1 → 0 e atinge 100 iterações sem convergir.</p></section></html>'
(SAIDA / 'dashboard.html').write_text(pagina, encoding='utf-8')
shutil.make_archive('resultados_atividade_3', 'zip', root_dir=SAIDA)
print('Dashboard:', SAIDA / 'dashboard.html')
print('Pacote: resultados_atividade_3.zip')